# GaussianDIR: paper loss, convergence and resources
Output-free source; execute an external copy after setting `GAUSSIANDIR_CONVERGENCE_REVIEW` to the generated report folder. This is one exploratory OASIS pair. Timing repetitions use the same seed. Saved SITReg fields supply reference accuracy, not inference timing.

In [ ]:
import os, json, csv, html
from pathlib import Path
from IPython.display import display, Markdown, Image, HTML
root = Path(os.environ['GAUSSIANDIR_CONVERGENCE_REVIEW']).resolve()
assert root.is_relative_to(Path(os.environ['ARTIFACT_ROOT']).resolve())
data = json.loads((root / 'comparison.json').read_text())
display(Markdown((root / 'report.md').read_text().split('## Review artifacts')[0]))

In [ ]:
for name in ('accuracy_time.png', 'optimization_traces.png', 'process_memory.png', 'regularity_time.png'):
    display(Image(filename=str(root / name)))

In [ ]:
def table(rows, columns):
    return '<table><thead><tr>' + ''.join('<th>'+html.escape(c)+'</th>' for c in columns) + '</tr></thead><tbody>' + ''.join('<tr>'+''.join('<td>'+html.escape(str(r.get(c,'')))+'</td>' for c in columns)+'</tr>' for r in rows) + '</tbody></table>'
label_rows = list(csv.DictReader((root / 'per_label.csv').open()))
selected = [r['job']['key'] for r in data['records'] if r['job']['run_kind'] == 'convergence'] + ['sitreg']
columns = ['label', 'eligible', 'voxel_counts.fixed', 'voxel_counts.moving', 'voxel_counts.warped', 'dice', 'hd95', 'asd_symmetric', 'asd_fixed_to_warped', 'asd_warped_to_fixed', 'exclusion_reason']
for key in selected:
    rows = [r for r in label_rows if r['key'] == key]
    assert len(rows) == 35
    display(Markdown('### All 35 labels: ' + key))
    display(HTML(table(rows, columns)))

In [ ]:
display(Markdown('## Full and foreground regularity of selected endpoints'))
for r in data['records']:
    if r['job']['run_kind'] != 'convergence': continue
    display(Markdown('### ' + r['job']['variant']))
    rows = [{'metric': k, 'value': v} for k,v in r['metrics']['regularity'].items() if not isinstance(v,dict)]
    display(HTML(table(rows, ['metric','value'])))
display(Markdown('## Timing repeats and descriptive speed/accuracy frontier'))
for filename in ('timing_repeats_summary.csv', 'descriptive_frontier.csv'):
    rows = list(csv.DictReader((root / filename).open()))
    display(Markdown('### ' + filename))
    display(HTML(table(rows, list(rows[0]))))
display(Markdown('All checkpoint, terminal, repeat, energy and per-label results are retained in the companion CSV/JSON files. No labels influence stopping or selection.'))